# 🌀 Cuando la recta no sirve: regresión polinomial (versión sencilla)
**Curso de IA · Bloque de Machine Learning · Sesión 5 (versión básica)**

---

## El problema de hoy: el consumo de un coche

¿Cuánto gasta un coche según la velocidad a la que va?

- A velocidades **muy bajas** el motor trabaja mal → gasta **mucho**.
- A velocidad **media** (unos 75 km/h) → gasta **lo mínimo**.
- A velocidades **muy altas** el aire frena el coche → gasta **mucho** otra vez.

Si lo dibujamos, sale una **"U"**. Y **una recta no puede dibujar una U**.

## La idea: añadir una columna nueva

La regresión lineal dibuja rectas con esta fórmula:

```
consumo = b + a × velocidad
```

Para que pueda hacer **curvas**, le damos un ingrediente más: la velocidad **al cuadrado**:

```
consumo = b + a1 × velocidad + a2 × velocidad²
```

Con ese truco, la recta se convierte en una **curva en forma de parábola**. Y a esto lo llamamos **regresión polinomial**.

> 💡 No es un modelo nuevo: sigue siendo **la misma regresión lineal**; lo único que hemos hecho es **añadir una columna** (`velocidad²`).

## El peligro: la curva demasiado "nerviosa"

Podemos añadir `velocidad³`, `velocidad⁴`... y la curva será cada vez **más flexible**. Pero si es **demasiado flexible**, dejará de aprender el patrón y empezará a **memorizar el ruido**. Hoy aprenderemos a **detectarlo**.

## Los pasos de hoy

| Apartado | Paso | En una frase |
|---|------|--------------|
| 1-2 | Herramientas y datos | Librerías y CSV |
| 3 | Cargar y dibujar | ¿Qué forma tienen los datos? |
| 4 | Separar entrenamiento y prueba | Guardar datos "para el examen" |
| 5 | Probar con una recta | ¿Por qué falla? |
| 6 | **Añadir `velocidad²`** | Convertir la recta en curva |
| 7 | `PolynomialFeatures` | El atajo para crear las columnas |
| 8 | **¿Qué grado elegir?** | Demasiado simple frente a demasiado nervioso |
| 9 | Usar el modelo | ¿A qué velocidad se gasta menos? |
| 10 | Resumen | La receta |

## Cómo usar este cuaderno

Ejecuta cada celda con **Shift + Enter**, **en orden**. Si algo se estropea: *Entorno de ejecución → Reiniciar y ejecutar todo*. Antes de ejecutar, **lee y piensa qué va a salir**.

> 👩‍🏫 **Nota para el profesor**
>
> Versión **simplificada** del cuaderno de regresión polinomial. Se han quitado: la **validación cruzada**, **Ridge**, el análisis de **extrapolación**, el caso con **varias variables** e interacciones, y los residuos. Solo se usa **una variable** (la velocidad) para poder **dibujar siempre** lo que ocurre.
>
> **Duración orientativa: unas 3 horas** con los ejercicios.

## 1. Preparar las herramientas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Todo listo ✅")

## 2. Crear el fichero de datos

Esta celda fabrica `consumo.csv` con **150 viajes**:

| Columna | Significado |
|---|---|
| `velocidad` | Velocidad media del viaje (km/h) |
| `consumo` | Litros gastados cada 100 km ← **lo que queremos predecir** |

> 💡 No hace falta entender el código de esta celda.

In [ ]:
rng = np.random.default_rng(5)
n = 150

velocidad = rng.uniform(20, 140, n).round(0)
consumo = (4.5 + 0.0011 * (velocidad - 75) ** 2 + rng.normal(0, 0.35, n)).round(2)

pd.DataFrame({"velocidad": velocidad, "consumo": consumo}).to_csv("consumo.csv", index=False)
print("✅ Fichero 'consumo.csv' creado")

## 3. Cargar y dibujar

**Siempre dibujar antes de modelar.**

In [ ]:
df = pd.read_csv("consumo.csv")
print("Filas y columnas:", df.shape)
df.head()

In [ ]:
plt.scatter(df["velocidad"], df["consumo"], alpha=0.6)
plt.xlabel("Velocidad (km/h)")
plt.ylabel("Consumo (litros / 100 km)")
plt.title("Consumo según la velocidad")
plt.show()

👀 **Se ve una "U" clarísima:** consumo alto a velocidades bajas, mínimo en el centro y alto otra vez a velocidades altas.

### Una advertencia: la correlación puede engañar

La **correlación** mide solo relaciones **en línea recta**. Veamos qué dice aquí:

In [ ]:
df.corr()["consumo"].round(2)

🚨 La correlación entre velocidad y consumo es **baja**, y sin embargo **la relación es muy fuerte** (la U es evidente). Porque la parte izquierda "baja" y la derecha "sube", y se compensan.

> 🧠 **Una correlación baja NO significa "no hay relación"**. Significa "no hay relación **en línea recta**". Por eso hay que **dibujar los datos**.

## 4. Separar entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["velocidad"]]      # doble corchete: tiene que ser una tabla
y = df["consumo"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Para entrenar:", len(X_train), "viajes | Para el examen:", len(X_test), "viajes")

## 5. Primer intento: una recta

Probamos la regresión lineal de siempre y miramos cómo le va, con el **R²** (una nota de 0 a 1: 0 = no mejora a predecir siempre la media, 1 = perfecto).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

recta = LinearRegression().fit(X_train, y_train)
pred_recta = recta.predict(X_test)

print(f"Recta  ->  R² = {r2_score(y_test, pred_recta):.2f}   |   error medio = {mean_absolute_error(y_test, pred_recta):.2f} litros")

In [ ]:
xs = pd.DataFrame({"velocidad": np.linspace(20, 140, 200)})

plt.scatter(X_train["velocidad"], y_train, alpha=0.5, label="Viajes (entrenamiento)")
plt.plot(xs["velocidad"], recta.predict(xs), color="red", linewidth=3, label="Recta")
plt.xlabel("Velocidad (km/h)")
plt.ylabel("Consumo (litros / 100 km)")
plt.title("Una recta intentando ajustar una U")
plt.legend()
plt.show()

👀 **La recta es casi inútil:** pasa por el medio de la nube sin captar su forma. El R² es muy bajo (casi como predecir siempre la media).

## 6. El truco: añadir la columna `velocidad²`

Creamos una **columna nueva** que sea la velocidad multiplicada por sí misma, y entrenamos la regresión lineal **con las dos columnas**:

In [ ]:
X_train2 = X_train.copy()
X_train2["velocidad2"] = X_train2["velocidad"] ** 2

X_test2 = X_test.copy()
X_test2["velocidad2"] = X_test2["velocidad"] ** 2

X_train2.head()

In [ ]:
curva = LinearRegression().fit(X_train2, y_train)
pred_curva = curva.predict(X_test2)

print(f"Curva (grado 2)  ->  R² = {r2_score(y_test, pred_curva):.2f}   |   error medio = {mean_absolute_error(y_test, pred_curva):.2f} litros")

In [ ]:
xs2 = xs.copy()
xs2["velocidad2"] = xs2["velocidad"] ** 2

plt.scatter(X_train["velocidad"], y_train, alpha=0.5, label="Viajes (entrenamiento)")
plt.plot(xs["velocidad"], recta.predict(xs), color="red", linestyle="--", linewidth=2, label="Recta")
plt.plot(xs["velocidad"], curva.predict(xs2), color="green", linewidth=3, label="Curva (con velocidad²)")
plt.xlabel("Velocidad (km/h)")
plt.ylabel("Consumo (litros / 100 km)")
plt.title("¡Ahora sí!")
plt.legend()
plt.show()

👀 **Espectacular:** solo con añadir la columna `velocidad²`, el modelo **dibuja la U** y el R² pasa de casi 0 a un valor **muy alto**. **No hemos cambiado de modelo**: seguimos usando `LinearRegression`. Solo le hemos dado **una columna más**.

## 7. El atajo: `PolynomialFeatures`

Crear columnas a mano (`velocidad²`, `velocidad³`, `velocidad⁴`...) es pesado. scikit-learn tiene una herramienta que **las crea todas**: `PolynomialFeatures(degree=3)` crea `velocidad`, `velocidad²` y `velocidad³`.

El **grado** (`degree`) es la **potencia más alta**:

| Grado | Forma de la curva |
|---|---|
| 1 | Una **recta** |
| 2 | Una **parábola** (una "U" o una "∩") |
| 3 | Una curva con hasta **2 giros** |
| 10 | Una curva con **muchos giros** |

Lo juntamos todo en un `make_pipeline` (una "cadena" de pasos que se ejecutan en orden):

```
velocidad  →  [StandardScaler]  →  [PolynomialFeatures]  →  [LinearRegression]  →  consumo
```

> 📌 El `StandardScaler` pone la velocidad en una escala pequeña. Sin él, al elevar 140 a la potencia 10 saldrían números **gigantescos** que desestabilizan los cálculos.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

def modelo_polinomial(grado):
    return make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=grado, include_bias=False),
        LinearRegression(),
    )

m2 = modelo_polinomial(2).fit(X_train, y_train)
print(f"Grado 2  ->  R² = {r2_score(y_test, m2.predict(X_test)):.2f}")

## 8. ¿Qué grado elegir? Demasiado simple frente a demasiado nervioso

Si el grado 2 va tan bien, ¿por qué no usar grado 10? Probémoslo. Para cada grado del 1 al 10 medimos el **R² en el entrenamiento** (los datos que ha estudiado) y **en la prueba** (el examen):

In [ ]:
filas = []
for grado in range(1, 11):
    m = modelo_polinomial(grado).fit(X_train, y_train)
    filas.append({
        "grado": grado,
        "R² entrenamiento": round(r2_score(y_train, m.predict(X_train)), 3),
        "R² prueba (examen)": round(r2_score(y_test, m.predict(X_test)), 3),
    })

tabla = pd.DataFrame(filas).set_index("grado")
tabla

In [ ]:
plt.plot(tabla.index, tabla["R² entrenamiento"], marker="o", color="seagreen", label="Entrenamiento")
plt.plot(tabla.index, tabla["R² prueba (examen)"], marker="o", color="darkorange", label="Prueba (examen)")
plt.xlabel("Grado del polinomio")
plt.ylabel("R²")
plt.title("¿Qué grado funciona mejor?")
plt.legend()
plt.show()

👀 **La gráfica más importante de la sesión:**

| Zona | Qué pasa | Nombre |
|---|---|---|
| **Grado 1** | R² bajo **en los dos** | **Demasiado simple** (*infraajuste*) |
| **Grado 2** | R² alto **en los dos** | ✅ **Justo** |
| **Grados altos** | El de **entrenamiento sigue subiendo (o se queda igual)** pero el de **prueba deja de mejorar o empeora** | **Demasiado nervioso** (*sobreajuste*) |

**Idea clave:** al subir el grado, el modelo **siempre** se ajusta mejor a los datos **de entrenamiento** (tiene más libertad). Pero lo que importa es el **examen**. Si en el examen no mejora, esa complejidad extra **no sirve** (o **estorba**).

### Verlo dibujado: tres grados

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)

for eje, grado, titulo in zip(ejes, [1, 2, 10], ["Grado 1: demasiado simple", "Grado 2: justo", "Grado 10: demasiado nervioso"]):
    m = modelo_polinomial(grado).fit(X_train, y_train)
    eje.scatter(X_train["velocidad"], y_train, alpha=0.4, s=15)
    eje.plot(xs["velocidad"], m.predict(xs), color="red", linewidth=2.5)
    eje.set_title(titulo)
    eje.set_xlabel("Velocidad (km/h)")

ejes[0].set_ylabel("Consumo")
plt.tight_layout()
plt.show()

👀 **Mira el grado 10:** la curva intenta pasar cerca de cada punto y hace **ondulaciones sin sentido** (¿el consumo sube y baja sin motivo al cambiar de 60 a 63 km/h?). Está **memorizando el ruido**.

> 🎯 **Regla de oro:** entre dos modelos que rinden parecido, **elige el más simple**. Aquí, el **grado 2**.

> ⚠️ **Otro aviso:** un polinomio **solo es fiable dentro del rango de los datos** con los que se entrenó (aquí, de 20 a 140 km/h). Fuera de ese rango, **la curva se dispara** y no hay que fiarse.

## 9. Usar el modelo

### ¿Cuánto consume a 90 km/h y a 120 km/h?

In [ ]:
modelo_final = modelo_polinomial(2).fit(X_train, y_train)

nuevas = pd.DataFrame({"velocidad": [50, 90, 120]})
resultado = nuevas.copy()
resultado["consumo estimado (l/100 km)"] = modelo_final.predict(nuevas).round(2)
resultado

### ¿A qué velocidad se gasta menos?

Esta es una ventaja de tener una **curva**: podemos buscar su **punto más bajo**. Pedimos el consumo para cada velocidad de 20 a 140 y nos quedamos con la mínima:

In [ ]:
velocidades = pd.DataFrame({"velocidad": np.arange(20, 141)})
consumos = modelo_final.predict(velocidades)

i = int(np.argmin(consumos))
print(f"Consumo mínimo estimado: {consumos[i]:.2f} l/100 km, a {velocidades['velocidad'][i]} km/h")

plt.plot(velocidades["velocidad"], consumos, color="green", linewidth=2.5)
plt.scatter(velocidades["velocidad"][i], consumos[i], color="red", s=100, zorder=3, label="Mínimo")
plt.xlabel("Velocidad (km/h)")
plt.ylabel("Consumo estimado (l/100 km)")
plt.title("La velocidad más económica")
plt.legend()
plt.show()

Este tipo de pregunta ("¿qué valor de `x` hace **mínimo** o **máximo** el resultado?") es muy útil: la dosis de un medicamento que más efecto da, el precio que más vende...

## 10. Resumen

### La receta de la regresión polinomial

| Paso | Qué hacemos |
|---|---|
| 1 | **Dibujar** los datos: ¿forman una recta o una curva? |
| 2 | Probar con **una recta** y ver que falla |
| 3 | Añadir columnas de **potencias** (`x²`, `x³`...) con `PolynomialFeatures` |
| 4 | Probar varios **grados** y comparar **entrenamiento y prueba** |
| 5 | Elegir **el grado más pequeño que funcione bien** |

### Ideas clave

1. Una **correlación baja** no significa que no haya relación: puede ser **curva**.
2. La regresión polinomial **no es un modelo nuevo**: es una regresión lineal con **columnas extra** (`x²`, `x³`...).
3. **Grado bajo** = demasiado simple (infraajuste). **Grado alto** = demasiado nervioso (sobreajuste).
4. El resultado en el **entrenamiento** siempre mejora al subir el grado; **el que cuenta es el de la prueba**.
5. **Entre dos modelos parecidos, el más simple.**
6. **No usar el modelo fuera del rango** de los datos de entrenamiento.

### 📖 Mini glosario

| Palabra | Significado sencillo |
|---|---|
| **Grado** | La potencia más alta que usamos (`x²` = grado 2) |
| **Infraajuste** | El modelo es demasiado simple y no capta el patrón |
| **Sobreajuste** | El modelo es demasiado complejo y memoriza el ruido |
| **Pipeline** | Una cadena de pasos que se ejecutan en orden |

## ✍️ Ejercicios

**Nivel 1 – Para empezar**
1. Con el modelo de grado 2, ¿cuánto consume un coche a 30 km/h? ¿Y a 75 km/h? ¿Tiene sentido que sean tan distintos?
2. Mira la tabla de grados. ¿A partir de qué grado el R² de **prueba** deja de mejorar?
3. Cambia `random_state=42` por otro número. ¿Cambian los R²? ¿Cambia el mejor grado?

**Nivel 2 – Practicar**
4. Dibuja la curva del grado **3** y la del grado **5**. ¿Se parecen a la del grado 2?
5. Sube el ruido de los datos (en la fábrica, cambia `0.35` por `1.5`). ¿Qué pasa con el R² de todos los grados? ¿Y con el grado 10?
6. Pregunta por el consumo a **200 km/h** con el grado 2 y con el grado 10. ¿Cuál da un resultado más razonable? (Recuerda el aviso sobre el rango.)

**Nivel 3 – Reto**
7. Cambia la fórmula de la "fábrica" para que sea **cúbica**: añade `+ 0.00001 * (velocidad - 75) ** 3`. ¿Qué grado funciona ahora mejor?

---
### 🎯 ¿Y ahora qué?
Hemos visto que **elegir la complejidad del modelo** es clave. Los siguientes modelos (árboles, bosques, boosting) **aprenden formas complicadas por sí solos**, pero necesitan el mismo cuidado con el sobreajuste.